## Import Datas

In [6]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [4]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},#
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [7]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [8]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.1 MB/s eta 0:00:00


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [10]:
import optuna
import warnings

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

In [15]:
def train_rf_with_optuna(X_train, y_train, X_test, y_test, best_params):
    """
    Train a RandomForest using Optuna best_params.
    """

    model = RandomForestRegressor(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        max_features=best_params["max_features"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train.ravel())

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def objective_for_rf(trial, X_train, y_train):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 5, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        n_jobs=1
    )

    score = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
    return -score.mean()

def predict_current_price_using_rf(option_type, ticher):
  for proxy in feature_combinations:
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    study = study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_rf(trial, X_train, y_train), n_trials=10)

    best_params = study.best_params
    #print(best_params)

    mae, rmse, nrmse = train_rf_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## Price options using xgboost

In [19]:
ticker = "GM"

In [20]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["SC_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [21]:
predict_current_price_using_rf("put", ticker)

[] for GM => (MAE=0.386; RMSE=0.815; ; NRMSE=0.363)
['GS_GM'] for GM => (MAE=0.414; RMSE=0.839; ; NRMSE=0.373)
['SC_GM'] for GM => (MAE=0.423; RMSE=0.886; ; NRMSE=0.394)
['VIX'] for GM => (MAE=0.422; RMSE=0.873; ; NRMSE=0.388)
['PCR'] for GM => (MAE=0.422; RMSE=0.871; ; NRMSE=0.387)
['GS_GM', 'SC_GM'] for GM => (MAE=0.441; RMSE=0.901; ; NRMSE=0.401)
['GS_GM', 'VIX'] for GM => (MAE=0.449; RMSE=0.912; ; NRMSE=0.405)
['GS_GM', 'PCR'] for GM => (MAE=0.449; RMSE=0.912; ; NRMSE=0.406)
['SC_GM', 'VIX'] for GM => (MAE=0.455; RMSE=0.952; ; NRMSE=0.423)
['SC_GM', 'PCR'] for GM => (MAE=0.462; RMSE=0.972; ; NRMSE=0.432)
['VIX', 'PCR'] for GM => (MAE=0.460; RMSE=0.959; ; NRMSE=0.426)
['GS_GM', 'SC_GM', 'VIX'] for GM => (MAE=0.471; RMSE=1.011; ; NRMSE=0.450)
['GS_GM', 'SC_GM', 'PCR'] for GM => (MAE=0.476; RMSE=1.021; ; NRMSE=0.454)
['SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.514; RMSE=1.072; ; NRMSE=0.477)
['SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.514; RMSE=1.072; ; NRMSE=0.477)
['GS_GM', 'SC_GM', 'VIX'

In [ ]:
# [] for F => (MAE=0.080; RMSE=0.192; ; NRMSE=0.226)  ['VIX'] for F => (MAE=0.100; RMSE=0.241; ; NRMSE=0.283)    ['PCR'] for F => (MAE=0.100; RMSE=0.241; ; NRMSE=0.283) ['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.102; RMSE=0.243; ; NRMSE=0.285)
# [] for GM => (MAE=0.386; RMSE=0.815; ; NRMSE=0.363) ['GS_GM'] for GM => (MAE=0.414; RMSE=0.839; ; NRMSE=0.373) ['GS_GM', 'SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.433; RMSE=0.912; ; NRMSE=0.406)

In [ ]:
predict_current_price_using_rf("call", ticker)

[] for GM => (MAE=0.675; RMSE=1.245; ; NRMSE=0.171)
['GS_GM'] for GM => (MAE=0.697; RMSE=1.252; ; NRMSE=0.172)
['SC_GM'] for GM => (MAE=0.718; RMSE=1.299; ; NRMSE=0.179)
['VIX'] for GM => (MAE=0.740; RMSE=1.370; ; NRMSE=0.188)
['PCR'] for GM => (MAE=0.718; RMSE=1.311; ; NRMSE=0.180)
['GS_GM', 'SC_GM'] for GM => (MAE=0.771; RMSE=1.352; ; NRMSE=0.186)
['GS_GM', 'VIX'] for GM => (MAE=0.774; RMSE=1.376; ; NRMSE=0.189)
['GS_GM', 'PCR'] for GM => (MAE=0.765; RMSE=1.368; ; NRMSE=0.188)
['SC_GM', 'VIX'] for GM => (MAE=0.793; RMSE=1.414; ; NRMSE=0.194)
['SC_GM', 'PCR'] for GM => (MAE=0.779; RMSE=1.386; ; NRMSE=0.190)
['VIX', 'PCR'] for GM => (MAE=0.807; RMSE=1.464; ; NRMSE=0.201)
['GS_GM', 'SC_GM', 'VIX'] for GM => (MAE=0.830; RMSE=1.492; ; NRMSE=0.205)


In [ ]:
# [] for F => (MAE=0.103; RMSE=0.182; ; NRMSE=0.098) ['GS_F'] for F => (MAE=0.125; RMSE=0.207; ; NRMSE=0.112) ['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.128; RMSE=0.212; ; NRMSE=0.115)